## NB4 — Initial conditions: Yj varies / shear centre (test298–test307)

**What changes:** jet centre Yj moves from shallow slope edge to deep side, staying within slope  
**Fixed:** Ljet=5 km, U=0.2 m/s, slope width=50 km (500 cells), slope height=400 m  
**dx=100 m, Lx=500 km, Ly=400 km**

The slope spans **175–225 km** (centred at 200 km, width=50 km).  
Yj steps from the **shallow edge (175 km)** to **just inside the deep edge (220 km)** in steps of 5 km.

| Exp | Yj (km) | Position on slope |
|-----|---------|-------------------|
| test298 | 175 km | -25 km from slope centre |
| test299 | 180 km | -20 km from slope centre |
| test300 | 185 km | -15 km from slope centre |
| test301 | 190 km | -10 km from slope centre |
| test302 | 195 km | -5 km from slope centre |
| test303 | 200 km | +0 km from slope centre |
| test304 | 205 km | +5 km from slope centre |
| test305 | 210 km | +10 km from slope centre |
| test306 | 215 km | +15 km from slope centre |
| test307 | 220 km | +20 km from slope centre |

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from netCDF4 import Dataset
import os

### Domain, grid and physical constants

In [2]:
# ── Domain (fixed for all experiments) ───────────────────────────────────────
Lx = 600e3    # m   500 km
Ly = 400e3    # m   400 km
dx = 500      # m
dy = 500      # m
nx = int(Lx / dx)   # 1000
ny = int(Ly / dy)   # 800
nxp = nx + 1
nyp = ny + 1

xh = np.linspace(dx/2, nx*dx - dx/2, nx)
yh = np.linspace(dy/2, ny*dy - dy/2, ny)
xq = np.linspace(0, nx*dx, nxp)
yq = np.linspace(0, ny*dy, nyp)

# Physical constants
f = -1e-5   # s^-1
g = 9.81    # m s^-2

# Bathymetry constants
shallow_depth = 100.0   # m  shelf
deep_depth    = 500.0   # m  basin
Ys            = 200e3   # m  slope centre

print(f'Grid: nx={nx}, ny={ny}  |  Domain: {Lx/1e3:.0f} x {Ly/1e3:.0f} km  |  dx={dx} m')

Grid: nx=1200, ny=800  |  Domain: 600 x 400 km  |  dx=500 m


### Helper functions

In [3]:
def make_fields(Yj, Ljet, U):
    """Velocity and eta fields for given jet parameters."""
    Xq_, Yh_u = np.meshgrid(xq, yh)
    Xh_, Yh_h = np.meshgrid(xh, yh)
    yprime_u   = (Yh_u - Yj) / Ljet
    yprime_h   = (Yh_h - Yj) / Ljet
    u_clean    = U * np.tanh(yprime_u)
    v          = np.zeros((nyp, nx))
    rng        = np.random.default_rng(seed=42)
    noise      = rng.standard_normal(u_clean.shape)
    decay      = 1.0 / np.cosh(yprime_u)**2
    u          = u_clean + 1e-3 * decay * noise
    eta_t      = -(f * U * Ljet / g) * np.log(np.cosh(yprime_h))
    eta_t     -= eta_t.mean()
    return u, v, eta_t


def make_bathy(slope_width_cells, slope_height_m):
    """2-D bathymetry. Slope centred at Ys.
    Even width -> centre on cell int(Ys/dy).
    Odd  width -> centre on cell int(Ys/dy)+1.
    Basin depth = shallow_depth + slope_height_m.
    """
    basin      = shallow_depth + slope_height_m
    c_idx      = int(Ys/dy) if slope_width_cells % 2 == 0 else int(Ys/dy) + 1
    ss         = c_idx - slope_width_cells // 2
    se         = ss + slope_width_cells
    profile    = np.zeros(ny)
    profile[:ss]   = shallow_depth
    profile[ss:se] = np.linspace(shallow_depth, basin, slope_width_cells)
    profile[se:]   = basin
    return np.tile(profile[:, np.newaxis], (1, nx)), ss, se


def write_netcdf(input_dir, u, v, eta_t, depth_2d):
    """Write init_vel.nc, init_eta.nc, ocean_topog.nc."""
    zl   = 1
    z    = np.array([-250.0])
    u_3d = u    [np.newaxis, :, :]
    v_3d = v    [np.newaxis, :, :]
    h_3d = eta_t[np.newaxis, :, :]

    # init_vel.nc
    nc = Dataset(os.path.join(input_dir, 'init_vel.nc'), 'w', format='NETCDF4')
    for dim, sz in [('zl',zl),('yh',ny),('xh',nx),('yq',nyp),('xq',nxp)]:
        nc.createDimension(dim, sz)
    for name, data, dims in [('zl',z,('zl',)),('xh',xh,('xh',)),
                               ('yh',yh,('yh',)),('xq',xq,('xq',)),('yq',yq,('yq',))]:
        v0=nc.createVariable(name,'f4',dims); v0[:]=data
    nc['zl'].axis='Z'; nc['zl'].long_name='depth to layer'
    for n_ in ['xh','xq']:
        nc[n_].axis='X'; nc[n_].long_name=f'{n_[1]}-point longitude'; nc[n_].units='meters'
    for n_ in ['yh','yq']:
        nc[n_].axis='Y'; nc[n_].long_name=f'{n_[1]}-point latitude';  nc[n_].units='meters'
    uv=nc.createVariable('u','f4',('zl','yh','xq')); uv[:]=u_3d
    uv.units='m s-1'; uv.long_name='Eastward velocity'
    uv.standard_name='eastward_sea_water_velocity'
    vv=nc.createVariable('v','f4',('zl','yq','xh')); vv[:]=v_3d
    vv.units='m s-1'; vv.long_name='Northward velocity'
    vv.standard_name='northward_sea_water_velocity'
    nc.regrid_method='bilinear'; nc.close()

    # init_eta.nc
    nc = Dataset(os.path.join(input_dir, 'init_eta.nc'), 'w', format='NETCDF4')
    nc.createDimension('yh', ny); nc.createDimension('xh', nx)
    v0=nc.createVariable('xh','f4',('xh',)); v0[:]=xh
    v0.axis='X'; v0.long_name='h-point longitude'; v0.units='meters'
    v0=nc.createVariable('yh','f4',('yh',)); v0[:]=yh
    v0.axis='Y'; v0.long_name='h-point latitude';  v0.units='meters'
    hv=nc.createVariable('eta_t','f4',('yh','xh')); hv[:]=h_3d
    hv.units='m'; hv.long_name='Free surface height anomaly'
    hv.standard_name='sea_floor_depth_below_sea_surface'; nc.close()

    # ocean_topog.nc
    nc = Dataset(os.path.join(input_dir, 'ocean_topog.nc'), 'w', format='NETCDF4')
    nc.createDimension('yh', ny); nc.createDimension('xh', nx)
    v0=nc.createVariable('xh','f4',('xh',)); v0[:]=xh
    v0.long_name='t-cell center x-location'; v0.units='meters'
    v0=nc.createVariable('yh','f4',('yh',)); v0[:]=yh
    v0.long_name='t-cell center y-location'; v0.units='meters'
    dv=nc.createVariable('depth','f4',('yh','xh')); dv[:]=depth_2d
    dv.units='m'; dv.long_name='ocean bottom depth'
    dv.standard_name='sea_floor_depth_below_geoid'; nc.close()

### Experiment definitions

In [4]:
base_dir          = '/scratch/nm03/ae7501/mom6_input_directories/idealized'
Ljet              = 5e3     # m  (5 km)
U                 = 0.2     # m/s
fixed_width_cells = 100     # cells = 50 km at dx=100m
fixed_height_m    = 400.0   # m  (shelf 100m -> basin 500m)

# Yj steps from shallow slope edge (175 km) to just inside deep edge (220 km)
# Robust fix — use actual slope cell centres
slope_yh = [dy/2 + i*dy for i in range(ny)]
ss = 350  # slope_start index
# pick 10 evenly spaced indices within the slope [350, 450)
yj_indices = np.linspace(350, 449, 10, dtype=int)
Yj_values_km = [round(slope_yh[i]/1e3, 3) for i in yj_indices]

experiments = [
    dict(exp_num=308+i, Ljet=Ljet, U=U,
         Yj=Yj_km*1e3, Yj_km=Yj_km,
         slope_w_cells=fixed_width_cells,
         slope_h_m=fixed_height_m)
    for i, Yj_km in enumerate(Yj_values_km)]

print(f'{len(experiments)} experiments: test{experiments[0]["exp_num"]} – test{experiments[-1]["exp_num"]}')
print(f'Slope: 175–225 km  |  Yj range: {Yj_values_km[0]}–{Yj_values_km[-1]} km')

10 experiments: test308 – test317
Slope: 175–225 km  |  Yj range: 175.25–224.75 km


### Main loop

In [6]:
for exp in experiments:
    exp_name  = f'test{exp["exp_num"]}'
    input_dir = os.path.join(base_dir, exp_name, 'INPUT')
    os.makedirs(input_dir, exist_ok=True)

    # Every experiment has a different Yj so fields are always recomputed
    u, v, eta_t = make_fields(exp['Yj'], exp['Ljet'], exp['U'])
    depth_2d, ss, se = make_bathy(exp['slope_w_cells'], exp['slope_h_m'])
    write_netcdf(input_dir, u, v, eta_t, depth_2d)

    offset = exp['Yj_km'] - 200
    print(f'{exp_name}: Yj={exp["Yj_km"]}km ({offset:+d}km from centre)  '
          f'slope=[{ss},{se})  -> {input_dir}')

print(f"\n{'='*60}\nDone. {len(experiments)} experiments written.\n{'='*60}")

ValueError: Unknown format code 'd' for object of type 'float'

### velocity profiles shifting across slope

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: velocity profiles
ax = axes[0]
colors = plt.cm.RdBu(np.linspace(0.1, 0.9, len(experiments)))
Xq_, Yh_u_ = np.meshgrid(xq, yh)
for i, exp in enumerate(experiments):
    yprime = (Yh_u_ - exp['Yj']) / exp['Ljet']
    u_1d   = (exp['U'] * np.tanh(yprime))[:, nx//2]
    ax.plot(yh/1e3, u_1d, color=colors[i], label=f'Yj={exp["Yj_km"]}km')
ax.axvspan(175, 225, color='wheat', alpha=0.4, label='slope (175–225 km)')
ax.axvline(200, color='k', ls='--', lw=0.8, label='slope centre')
ax.set_xlim(150, 250); ax.set_xlabel('y (km)'); ax.set_ylabel('u (m/s)')
ax.set_title('Velocity profiles — Yj shifting across slope')
ax.legend(fontsize=7, ncol=2); ax.grid(True, alpha=0.3)

# Right: bathymetry (same for all)
ax = axes[1]
d2d, ss, se = make_bathy(fixed_width_cells, fixed_height_m)
ax.fill_between(yh/1e3, 0, -d2d[:,0], color='steelblue', alpha=0.2)
ax.fill_between(yh/1e3, -d2d[:,0], -600, color='saddlebrown', alpha=0.5)
ax.plot(yh/1e3, -d2d[:,0], 'k', lw=2)
for i, exp in enumerate(experiments):
    ax.axvline(exp['Yj_km'], color=colors[i], lw=1, ls='--', alpha=0.7)
ax.set_xlim(150, 250); ax.set_ylim(-600, 10); ax.invert_yaxis()
ax.set_xlabel('y (km)'); ax.set_ylabel('Depth (m)')
ax.set_title('Bathymetry with Yj positions marked')
ax.grid(True, alpha=0.3)

plt.suptitle('NB4 — Yj varies across slope (Ljet=5km, U=0.2)', fontsize=12)
plt.tight_layout(); plt.show()